# OpenSky Collector — Schritt für Schritt

Dieses Notebook erklärt den Collector und führt ihn aus.

**Architektur:**
```
OpenSky Network API  →  get_departures() / get_arrivals()  →  build_document()  →  MongoDB Atlas (airline_landing.opensky_raw)
```

**Wichtig:** Der Collector muss lokal laufen — die OpenSky OAuth2-Auth (`auth.opensky-network.org`) ist nur vom Mac aus erreichbar (externe VMs geblockt). Siehe ADR 004 / ADR 005.

**Voraussetzung:** `.env` im **Projekt-Root** (`airline-data-platform/.env`) mit:
```
OPENSKY_CLIENT_ID=...
OPENSKY_CLIENT_SECRET=...
MONGO_URI=mongodb+srv://airline-collector-rw:<password>@mongo-mk1.ptb1k2b.mongodb.net/airline_landing
MONGO_DB=airline_landing
```
Dieser Collector schreibt Daten — `MONGO_URI` muss auf `airline-collector-rw` (Schreibzugriff) zeigen.
Credentials: `.env` ist gitignored, nie committen.

## Schritt 1 — Imports und Pfad-Setup

Der Collector liegt in `collectors/opensky_collector.py`.  
Der OpenSky-Client liegt in `opensky_api/client.py`.  
Der MongoDB-Connector liegt in `db/mongo/connector.py`.  
Damit Python diese Module findet, muss das aktuelle Verzeichnis (`03-data-collection/`) im Suchpfad sein.

In [ ]:
import sys
from pathlib import Path

notebook_dir = Path(".").resolve()
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

print(f"Arbeitsverzeichnis: {notebook_dir}")

## Schritt 2 — Zeitfenster konfigurieren

Die OpenSky Flights-API liefert abgeschlossene Flüge in einem Zeitfenster (`begin` / `end` als Unix-Timestamp).  
Wir schauen standardmäßig 24 Stunden zurück. Das Fenster darf max. 7 Tage umfassen.

| Parameter | Bedeutung |
|---|---|
| `begin` | Fenster-Start (Unix-Timestamp) |
| `end` | Fenster-Ende (Unix-Timestamp, typischerweise `now`) |
| `airport` | ICAO-Code, z.B. `EDDB` (Berlin Brandenburg) |

In [ ]:
import time
from datetime import datetime, timezone

AIRPORT = "EDDB"
LOOK_BACK_HOURS = 24

end = int(time.time())
begin = end - LOOK_BACK_HOURS * 3600

print(f"Airport:  {AIRPORT}")
print(f"Begin:    {begin}  ({datetime.fromtimestamp(begin, tz=timezone.utc).isoformat()})")
print(f"End:      {end}  ({datetime.fromtimestamp(end, tz=timezone.utc).isoformat()})")
print(f"Fenster:  {LOOK_BACK_HOURS} Stunden")

## Schritt 3 — API abfragen (Mock-Modus)

Zuerst mit `use_mock=True` — kein Login, keine Netzwerkverbindung nötig.  
Die Mock-Daten in `opensky_api/mock_data.py` spiegeln die echte API-Antwortstruktur.

Wir fragen **Abflüge** und **Ankünfte** für `EDDB` ab — je eine separate API-Anfrage.

In [ ]:
from opensky_api.client import OpenSkyClient

client_mock = OpenSkyClient(use_mock=True)

departures = client_mock.get_departures(AIRPORT, begin, end)
arrivals   = client_mock.get_arrivals(AIRPORT, begin, end)

print(f"Abflüge:  {len(departures)} Flüge")
print(f"Ankünfte: {len(arrivals)} Flüge")
print()
print("Erster Abflug (raw):")
departures[0] if departures else print("(keine Daten)")

## Schritt 4 — Flüge als DataFrame anschauen

Die API liefert pro Flug ein Dict. Wichtige Felder:

| Feld | Bedeutung |
|---|---|
| `icao24` | Transponder-Adresse des Flugzeugs |
| `callsign` | Rufzeichen (z.B. `EWG1R`) — kann Leerzeichen am Ende haben |
| `firstSeen` | Unix-Timestamp Abflug |
| `lastSeen` | Unix-Timestamp Landung |
| `estDepartureAirport` | ICAO-Code Abflughafen |
| `estArrivalAirport` | ICAO-Code Zielflughafen |
| `estDepartureAirportHorizDistance` | Horizontale Distanz zum Abflughafen (m) |

In [ ]:
import pandas as pd

df_dep = pd.DataFrame(departures)
df_arr = pd.DataFrame(arrivals)

print(f"Abflüge — Shape: {df_dep.shape}")

show = [c for c in ['icao24', 'callsign', 'estDepartureAirport', 'estArrivalAirport', 'firstSeen', 'lastSeen'] if c in df_dep.columns]
df_dep[show]

## Schritt 5 — Dokument bauen

`build_document()` verpackt die rohe Flug-Liste in das MongoDB-Format.  
Das `flights`-Array bleibt **unverändert** — keine Transformation, kein Flattening.  
Das ist bewusst: MongoDB ist die Raw Landing Zone; der ETL-Layer normalisiert später.

In [ ]:
from collectors.opensky_collector import build_document

doc_dep = build_document("departures", AIRPORT, begin, end, departures)
doc_arr = build_document("arrivals",   AIRPORT, begin, end, arrivals)

print("Departures-Dokument (ohne flights-Array):")
for k, v in doc_dep.items():
    if k != "flights":
        print(f"  {k}: {v}")
print(f"  flights: [{len(doc_dep['flights'])} Flug-Dicts]")

## Schritt 6 — Echte API (optional)

Für den Live-Abruf `use_mock=False` setzen. Voraussetzung: `.env` mit gültigen Credentials.

```python
client_live = OpenSkyClient(use_mock=False)
departures_live = client_live.get_departures(AIRPORT, begin, end)
```

Die OAuth2-Token-URL ist `https://auth.opensky-network.org/...` — die VM kann diese URL nicht erreichen,
deshalb läuft dieser Collector **nur lokal** (siehe ADR 004).

## Schritt 7 — Mit MongoDB verbinden

`from_env()` liest `MONGO_URI` und `MONGO_DB` aus der `.env`-Datei im Projekt-Root.  
MongoDB läuft auf **MongoDB Atlas** (`mongo-mk1`, eu-central-1) — Verbindung via SRV-URI.

In [ ]:
from db.mongo.connector import from_env

db = from_env().connect()
print("Verbunden.")

## Schritt 8 — Snapshots in MongoDB schreiben

`insert_raw()` ruft intern `collection.insert_one(doc)` auf.  
Pro Collector-Lauf entstehen **zwei Dokumente**: eines für Abflüge, eines für Ankünfte.

In [ ]:
id_dep = db.insert_raw("opensky_raw", doc_dep)
id_arr = db.insert_raw("opensky_raw", doc_arr)

print(f"Departures gespeichert: _id={id_dep}")
print(f"Arrivals   gespeichert: _id={id_arr}")

total = db.collection("opensky_raw").count_documents({})
print(f"\nDokumente gesamt in opensky_raw: {total}")

## Schritt 9 — Daten zurücklesen und prüfen

Kurze Verifikation: die letzten zwei Dokumente aus `opensky_raw` abrufen und Metadaten anzeigen.

In [ ]:
col = db.collection("opensky_raw")

print("Letzte 2 Dokumente in opensky_raw:")
for doc in col.find({}, {"flights": 0}).sort("collected_at", -1).limit(2):
    doc["_id"] = str(doc["_id"])
    print(doc)

## Schritt 10 — Verbindung schließen

In der Praxis läuft der Collector als Script (kein Notebook nötig):  
```bash
cd 03-data-collection
python collectors/opensky_collector.py --hours 24
python collectors/opensky_collector.py --mock   # ohne Credentials
```

In [ ]:
db.close()
print("Verbindung geschlossen.")